# 02 岭回归 Ridge Regression

依赖安装说明：`pip install numpy matplotlib scikit-learn`

岭回归是在线性回归基础上加了 L2 正则化的模型。它特别适合特征很多、特征彼此相关、普通线性回归系数容易变得很大的情况。


## 1. 数学逻辑

普通线性回归只关心预测误差：

$$\frac{1}{n}\sum_i(y_i-\hat y_i)^2$$

岭回归额外惩罚过大的权重：

$$L(w)=\frac{1}{n}\sum_{i=1}^{n}(y_i-X_iw)^2 + \lambda\sum_{j=1}^{d}w_j^2$$

`lambda` 越大，模型越不愿意使用很大的系数。

直觉：如果很多特征都能解释目标，岭回归会把权重分散得更平滑，而不是让某几个系数特别极端。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

n = 120
x1 = np.random.normal(size=n)
x2 = x1 + np.random.normal(scale=0.08, size=n)  # 和 x1 高度相关
x3 = np.random.normal(size=n)
X = np.column_stack([x1, x2, x3])
y = 3 * x1 + 3 * x2 + 0.2 * x3 + np.random.normal(scale=1.0, size=n)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [ ]:
# 从零实现：岭回归的闭式解
# w = (X^T X + alpha I)^(-1) X^T y

def add_bias(X):
    return np.column_stack([np.ones(len(X)), X])

def ridge_closed_form(X, y, alpha):
    Xb = add_bias(X)
    penalty = np.eye(Xb.shape[1])
    penalty[0, 0] = 0  # 截距项通常不正则化
    return np.linalg.solve(Xb.T @ Xb + alpha * penalty, Xb.T @ y)

for alpha in [0.0, 1.0, 10.0, 100.0]:
    coef = ridge_closed_form(X_train_s, y_train, alpha)
    pred = add_bias(X_test_s) @ coef
    print(f'alpha={alpha:5.1f} | bias={coef[0]: .3f} | weights={np.round(coef[1:], 3)} | MSE={mean_squared_error(y_test, pred):.3f}')


In [ ]:
# sklearn 实战：对比普通线性回归和 Ridge
ols = LinearRegression().fit(X_train_s, y_train)
ridge = Ridge(alpha=10.0).fit(X_train_s, y_train)

for name, model in [('LinearRegression', ols), ('Ridge(alpha=10)', ridge)]:
    pred = model.predict(X_test_s)
    print(name)
    print('  weights:', np.round(model.coef_, 3))
    print('  MSE:', round(mean_squared_error(y_test, pred), 3))

alphas = np.logspace(-3, 3, 60)
coefs = []
for a in alphas:
    coefs.append(Ridge(alpha=a).fit(X_train_s, y_train).coef_)
coefs = np.array(coefs)

plt.plot(alphas, coefs)
plt.xscale('log')
plt.title('alpha 越大，Ridge 系数越收缩')
plt.xlabel('alpha')
plt.ylabel('coefficient')
plt.show()


## 2. 评价和使用建议

- 回归任务仍然看 `MSE`、`MAE`、`R^2`。
- `alpha` 是关键超参数，通常用交叉验证选择。
- Ridge 不会把系数压成严格的 0；它更像“让所有系数变小”。

## 3. 常见误区

- 使用正则化前通常要标准化特征，否则不同量纲的特征会受到不公平惩罚。
- `alpha` 不是越大越好；太大时会欠拟合。
- Ridge 适合相关特征很多的情况，但不适合直接做特征选择。

## 4. 小实验

- 改 `alpha`，观察系数曲线。
- 增加无关噪声特征，观察 Ridge 和普通线性回归的差别。
- 去掉标准化，看看系数解释是否变得混乱。
